# Part 4 — Neural Network, Ablation, XAI & Inference

**Role: Person 4 — Zeina**

This notebook is designed to work for **both Kaggle submission and the GitHub repository**.

It now matches Person 2's pushed feature-engineering section exactly.

It covers:
- shallow feed-forward neural network
- multiple NN trials
- ROC-AUC, PR-AUC, and F1 evaluation
- feature-group ablation
- permutation importance
- global SHAP
- local SHAP
- inference on complete and partially-missing records

> The final run must use Person 2's leakage-safe `modeling_table` and exact `ALLOWED_FEATURES`.


## 1. Imports and reproducibility

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_recall_curve, roc_curve, confusion_matrix,
    classification_report
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

try:
    import shap
    SHAP_AVAILABLE = True
    print("SHAP:", shap.__version__)
except Exception as e:
    SHAP_AVAILABLE = False
    print("SHAP import failed:", e)


## 2. Load Person 2's final modeling table

Person 2's notebook builds an in-memory dataframe named `modeling_table` and also saves:

`/kaggle/working/outputs/modeling_table.csv`

This section first uses the in-memory table when Part 1 + Part 2 have already run in the same Kaggle notebook. Otherwise it searches for the saved CSV/Parquet file.


In [ ]:
import os

# Prefer the in-memory table created by Person 2 when all parts are merged
if "modeling_table" in globals() and isinstance(modeling_table, pd.DataFrame):
    df = modeling_table.copy()
    DATA_READY = True
    MODEL_TABLE_PATH = "in-memory modeling_table"
else:
    candidate_paths = [
        "/kaggle/working/outputs/modeling_table.csv",
        "/kaggle/input/modeling-table/modeling_table.csv",
        "./outputs/modeling_table.csv",
        "./data/processed/modeling_table.csv",
        "./modeling_table.csv",
    ]

    MODEL_TABLE_PATH = next((p for p in candidate_paths if os.path.exists(p)), None)

    if MODEL_TABLE_PATH is not None:
        df = pd.read_csv(MODEL_TABLE_PATH)
        DATA_READY = True
    else:
        DATA_READY = False
        df = None

if DATA_READY:
    print("Loaded modeling table from:", MODEL_TABLE_PATH)
    print("Shape:", df.shape)
    display(df.head())
else:
    print("No final modeling table found yet.")
    print("Run Person 1 + Person 2 first, or attach their modeling_table.csv, then Run All.")


## 3. Validate Person 2's modeling table

Person 2 defines:
- target: `podium`
- one row per `(raceId, driverId)`
- chronological field: `year`

The target must remain binary and must never be used as an input feature.


In [ ]:
if DATA_READY:
    target_col = "podium"
    year_col = "year"

    required = {target_col, year_col}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required Person 2 columns: {missing}")

    if not df[target_col].dropna().isin([0, 1]).all():
        raise ValueError("podium target must be binary 0/1")

    print("Target:", target_col)
    print("Year column:", year_col)
    display(df[target_col].value_counts(dropna=False).rename("count").to_frame())
    print("Podium prevalence:", round(df[target_col].mean(), 4))

    if {"raceId", "driverId"}.issubset(df.columns):
        duplicate_pairs = df.duplicated(["raceId", "driverId"]).sum()
        print("Duplicate (raceId, driverId) rows:", duplicate_pairs)
        if duplicate_pairs:
            raise ValueError("Modeling table is not unique at (raceId, driverId)")


## 4. Exact feature set from Person 2

Person 2 explicitly defines `ALLOWED_FEATURES`.  
This notebook uses that exact list so the neural network is trained on the same intended predictors.

Important:
- `podium` is the target only.
- `raceId`, `driverId`, and `constructorId` are not predictors.
- `circuitId` **is intentionally included** by Person 2 and is treated as a categorical feature here.


In [ ]:
PERSON2_ALLOWED_FEATURES = [
    "year",
    "circuitId",
    "grid_position",
    "qualifying_position",
    "is_home_race",
    "driver_prior_starts",
    "driver_prior_podium_rate",
    "driver_prior_mean_finish",
    "constructor_prior_podium_rate",
    "constructor_prior_mean_points",
    "driver_circuit_prior_starts",
    "driver_circuit_prior_podium_rate",
    "driver_prior_standing_points",
    "driver_prior_standing_position",
    "constructor_prior_standing_points",
    "constructor_prior_standing_position",
]

if DATA_READY:
    missing_features = [c for c in PERSON2_ALLOWED_FEATURES if c not in df.columns]
    if missing_features:
        raise ValueError(f"Missing Person 2 features: {missing_features}")

    feature_cols = PERSON2_ALLOWED_FEATURES.copy()

    print("Using exact Person 2 feature set:")
    print(feature_cols)
    print("Number of features:", len(feature_cols))

    assert "podium" not in feature_cols
    assert "positionOrder" not in feature_cols
    assert "points" not in feature_cols
    assert "status" not in feature_cols


## 5. Chronological train / validation / test split

We use the project companion's chronological split:
- Train: `year <= 2019`
- Validation: `2020–2021`
- Test: `year >= 2022`

`year` remains a model feature because Person 2 explicitly included it in `ALLOWED_FEATURES`; it is also used to define the temporal split.


In [ ]:
if DATA_READY:
    work = df.dropna(subset=[target_col, year_col]).copy()
    work[year_col] = pd.to_numeric(work[year_col], errors="coerce")
    work = work.dropna(subset=[year_col])

    train_df = work[work[year_col] <= 2019].copy()
    val_df   = work[work[year_col].between(2020, 2021)].copy()
    test_df  = work[work[year_col] >= 2022].copy()

    print("Train:", train_df.shape,
          f"years {int(train_df[year_col].min())}–{int(train_df[year_col].max())}")
    print("Validation:", val_df.shape,
          f"years {int(val_df[year_col].min())}–{int(val_df[year_col].max())}")
    print("Test:", test_df.shape,
          f"years {int(test_df[year_col].min())}–{int(test_df[year_col].max())}")

    assert len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0

    X_train_raw = train_df[feature_cols].copy()
    y_train = train_df[target_col].astype(int).to_numpy()

    X_val_raw = val_df[feature_cols].copy()
    y_val = val_df[target_col].astype(int).to_numpy()

    X_test_raw = test_df[feature_cols].copy()
    y_test = test_df[target_col].astype(int).to_numpy()

    print("\nPodium rate by split:")
    print("Train:", round(y_train.mean(), 4))
    print("Validation:", round(y_val.mean(), 4))
    print("Test:", round(y_test.mean(), 4))


## 6. Preprocessing

For the neural network:
- `circuitId` is treated as a **categorical** feature and one-hot encoded.
- remaining numeric features use median imputation and standardization.
- categorical missing values use most-frequent imputation.
- preprocessing is fit on the **training set only**.

This preserves Person 2's feature list while avoiding treating circuit identifiers as ordered numeric values.


In [ ]:
CATEGORICAL_RAW = ["circuitId"]

def build_preprocessor(X):
    categorical_cols = [c for c in CATEGORICAL_RAW if c in X.columns]
    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    transformers = [("num", num_pipe, numeric_cols)]

    if categorical_cols:
        transformers.append(("cat", cat_pipe, categorical_cols))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    return preprocessor, numeric_cols, categorical_cols

if DATA_READY:
    # Treat circuitId as nominal rather than ordinal.
    for split_x in [X_train_raw, X_val_raw, X_test_raw]:
        split_x["circuitId"] = split_x["circuitId"].astype("string")

    preprocessor, numeric_cols, categorical_cols = build_preprocessor(X_train_raw)

    X_train = np.asarray(
        preprocessor.fit_transform(X_train_raw),
        dtype=np.float32
    )
    X_val = np.asarray(
        preprocessor.transform(X_val_raw),
        dtype=np.float32
    )
    X_test = np.asarray(
        preprocessor.transform(X_test_raw),
        dtype=np.float32
    )

    try:
        transformed_feature_names = preprocessor.get_feature_names_out().tolist()
    except Exception:
        transformed_feature_names = [f"x{i}" for i in range(X_train.shape[1])]

    print("Raw numerical features:", numeric_cols)
    print("Raw categorical features:", categorical_cols)
    print("Transformed dimensions:", X_train.shape[1])


## 7. Evaluation utilities

In [ ]:
def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    if len(thresholds) == 0:
        return 0.5
    f1s = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[int(np.nanargmax(f1s))])

def metric_dict(y_true, y_prob, threshold):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "Threshold": threshold,
    }

def plot_roc_pr(y_true, y_prob, title_prefix="Model"):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y_true, y_prob):.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title_prefix} — ROC Curve")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"AP={average_precision_score(y_true, y_prob):.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix} — Precision–Recall Curve")
    plt.legend()
    plt.show()


## 8. Shallow NN architectures

In [ ]:
def build_nn_trial_1(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(32, activation="relu"),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy")
    return model

def build_nn_trial_2(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation="relu"),
        Dropout(0.20),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy")
    return model

def train_nn(builder, X_train, y_train, X_val, y_val, verbose=0):
    model = builder(X_train.shape[1])
    callback = EarlyStopping(
        monitor="val_loss",
        patience=8,
        min_delta=1e-4,
        restore_best_weights=True,
    )
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=150,
        batch_size=32,
        callbacks=[callback],
        verbose=verbose,
    )
    return model, history

def plot_history(history, title):
    plt.figure(figsize=(7, 5))
    plt.plot(history.history["loss"], label="Train loss")
    plt.plot(history.history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Binary cross-entropy")
    plt.title(title)
    plt.legend()
    plt.show()


## 9. Train and compare NN trials

In [ ]:
if DATA_READY:
    model1, history1 = train_nn(build_nn_trial_1, X_train, y_train, X_val, y_val)
    model2, history2 = train_nn(build_nn_trial_2, X_train, y_train, X_val, y_val)

    plot_history(history1, "NN Trial 1 — Training History")
    plot_history(history2, "NN Trial 2 — Training History")

    val_prob_1 = model1.predict(X_val, verbose=0).ravel()
    val_prob_2 = model2.predict(X_val, verbose=0).ravel()

    thr1 = best_f1_threshold(y_val, val_prob_1)
    thr2 = best_f1_threshold(y_val, val_prob_2)

    r1 = metric_dict(y_val, val_prob_1, thr1)
    r2 = metric_dict(y_val, val_prob_2, thr2)

    nn_comparison = pd.DataFrame([
        {"Model": "NN Trial 1 (32-16)", **r1},
        {"Model": "NN Trial 2 (64-32 + dropout)", **r2},
    ]).sort_values(["PR-AUC", "F1"], ascending=False)

    display(nn_comparison)

    winner = nn_comparison.iloc[0]["Model"]
    if winner.startswith("NN Trial 1"):
        best_model, best_history, best_threshold = model1, history1, thr1
        best_label = "NN Trial 1"
    else:
        best_model, best_history, best_threshold = model2, history2, thr2
        best_label = "NN Trial 2"

    print("Selected using validation data only:", best_label)
    print("Chosen threshold:", round(best_threshold, 4))

    plot_roc_pr(y_val, best_model.predict(X_val, verbose=0).ravel(), f"{best_label} Validation")


## 10. Train / validation / unseen test results

In [ ]:
if DATA_READY:
    split_rows = []
    for split_name, Xs, ys in [
        ("Train", X_train, y_train),
        ("Validation", X_val, y_val),
        ("Test", X_test, y_test),
    ]:
        probs = best_model.predict(Xs, verbose=0).ravel()
        split_rows.append({"Split": split_name, **metric_dict(ys, probs, best_threshold)})

    final_nn_results = pd.DataFrame(split_rows)
    display(final_nn_results)

    test_prob = best_model.predict(X_test, verbose=0).ravel()
    test_pred = (test_prob >= best_threshold).astype(int)

    print(classification_report(y_test, test_pred, digits=4))
    display(pd.DataFrame(
        confusion_matrix(y_test, test_pred),
        index=["Actual 0", "Actual 1"],
        columns=["Pred 0", "Pred 1"]
    ))

    plot_roc_pr(y_test, test_prob, f"{best_label} Unseen Test")


## 11. Feature-group ablation

The groups below are defined directly from Person 2's engineered features rather than guessed from column names.

Each experiment removes one complete raw feature group and retrains the same selected NN architecture.


In [ ]:
if DATA_READY:
    feature_groups = {
        "grid": [
            "grid_position",
        ],
        "qualifying": [
            "qualifying_position",
        ],
        "driver_form": [
            "driver_prior_starts",
            "driver_prior_podium_rate",
            "driver_prior_mean_finish",
            "driver_circuit_prior_starts",
            "driver_circuit_prior_podium_rate",
        ],
        "constructor_form": [
            "constructor_prior_podium_rate",
            "constructor_prior_mean_points",
        ],
        "standings": [
            "driver_prior_standing_points",
            "driver_prior_standing_position",
            "constructor_prior_standing_points",
            "constructor_prior_standing_position",
        ],
        "context": [
            "year",
            "circuitId",
            "is_home_race",
        ],
    }

    used = sorted({c for cols in feature_groups.values() for c in cols})
    expected = sorted(feature_cols)

    print("Feature groups:")
    for group, cols in feature_groups.items():
        print(f"\n{group}:")
        for c in cols:
            print(" -", c)

    assert used == expected, (
        "Ablation groups do not cover Person 2's exact feature list. "
        "Update the groups before the final run."
    )


In [ ]:
def run_ablation(raw_train, y_train, raw_val, y_val, base_features, groups, selected_builder):
    rows = []

    def fit_eval(cols, label):
        prep, _, _ = build_preprocessor(raw_train[cols])
        Xt = np.asarray(prep.fit_transform(raw_train[cols]), dtype=np.float32)
        Xv = np.asarray(prep.transform(raw_val[cols]), dtype=np.float32)

        model, _ = train_nn(selected_builder, Xt, y_train, Xv, y_val, verbose=0)
        prob = model.predict(Xv, verbose=0).ravel()
        threshold = best_f1_threshold(y_val, prob)
        m = metric_dict(y_val, prob, threshold)

        rows.append({
            "Experiment": label,
            "Num raw features": len(cols),
            **m
        })
        return m

    full = fit_eval(base_features, "Full feature set")

    for group_name, group_cols in groups.items():
        remaining = [c for c in base_features if c not in set(group_cols)]
        if remaining and group_cols:
            fit_eval(remaining, f"Without {group_name}")

    out = pd.DataFrame(rows)
    out["Δ PR-AUC vs full"] = out["PR-AUC"] - full["PR-AUC"]
    out["Δ ROC-AUC vs full"] = out["ROC-AUC"] - full["ROC-AUC"]
    out["Δ F1 vs full"] = out["F1"] - full["F1"]
    return out

if DATA_READY:
    selected_builder = build_nn_trial_1 if best_label == "NN Trial 1" else build_nn_trial_2

    ablation_results = run_ablation(
        X_train_raw, y_train,
        X_val_raw, y_val,
        feature_cols,
        feature_groups,
        selected_builder
    )

    display(ablation_results.sort_values("PR-AUC", ascending=False))

    plot_df = ablation_results.set_index("Experiment")["PR-AUC"].sort_values()
    plt.figure(figsize=(9, 5))
    plot_df.plot(kind="barh")
    plt.xlabel("Validation PR-AUC")
    plt.title("Feature-Group Ablation")
    plt.tight_layout()
    plt.show()


## 12. Permutation importance

Importance here means **model reliance**, not causality.


In [ ]:
def permutation_importance_pr_auc(model, X, y, feature_names, repeats=3, seed=SEED):
    rng = np.random.default_rng(seed)
    baseline_prob = model.predict(X, verbose=0).ravel()
    baseline = average_precision_score(y, baseline_prob)

    rows = []
    for j, name in enumerate(feature_names):
        drops = []

        for _ in range(repeats):
            Xp = X.copy()
            rng.shuffle(Xp[:, j])

            score = average_precision_score(
                y,
                model.predict(Xp, verbose=0).ravel()
            )
            drops.append(baseline - score)

        rows.append({
            "Feature": name,
            "Importance (PR-AUC drop)": float(np.mean(drops)),
            "Std": float(np.std(drops)),
        })

    return pd.DataFrame(rows).sort_values(
        "Importance (PR-AUC drop)", ascending=False
    )

if DATA_READY:
    permutation_df = permutation_importance_pr_auc(
        best_model, X_val, y_val,
        transformed_feature_names,
        repeats=3
    )

    display(permutation_df.head(20))

    top_perm = permutation_df.head(20).sort_values("Importance (PR-AUC drop)")
    plt.figure(figsize=(9, 7))
    plt.barh(top_perm["Feature"], top_perm["Importance (PR-AUC drop)"])
    plt.xlabel("Decrease in validation PR-AUC after shuffling")
    plt.title("Permutation Importance — Top 20")
    plt.tight_layout()
    plt.show()


## 13. Global SHAP

In [ ]:
if DATA_READY and SHAP_AVAILABLE:
    rng = np.random.default_rng(SEED)

    bg_n = min(50, len(X_train))
    ex_n = min(60, len(X_val))

    bg_idx = rng.choice(len(X_train), size=bg_n, replace=False)
    ex_idx = rng.choice(len(X_val), size=ex_n, replace=False)

    X_bg = X_train[bg_idx]
    X_explain = X_val[ex_idx]

    def predict_fn(x):
        return best_model.predict(
            np.asarray(x, dtype=np.float32),
            verbose=0
        ).ravel()

    kernel_explainer = shap.KernelExplainer(predict_fn, X_bg)
    shap_values = kernel_explainer.shap_values(X_explain, nsamples=100)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values)
    X_explain_df = pd.DataFrame(
        X_explain,
        columns=transformed_feature_names
    )

    shap.summary_plot(
        shap_values,
        X_explain_df,
        show=False,
        max_display=20
    )
    plt.title("Global SHAP Summary — Neural Network")
    plt.tight_layout()
    plt.show()
elif DATA_READY:
    print("SHAP unavailable in this environment.")


## 14. Local SHAP explanation

In [ ]:
if DATA_READY and SHAP_AVAILABLE:
    local_i = 0
    local_values = shap_values[local_i]
    local_x = X_explain[local_i]
    local_base = float(
        np.asarray(kernel_explainer.expected_value).reshape(-1)[0]
    )

    local_exp = shap.Explanation(
        values=local_values,
        base_values=local_base,
        data=local_x,
        feature_names=transformed_feature_names,
    )

    probability = float(
        best_model.predict(
            local_x.reshape(1, -1),
            verbose=0
        ).ravel()[0]
    )

    print("Explained prediction probability:", round(probability, 4))

    shap.plots.waterfall(
        local_exp,
        max_display=15,
        show=False
    )
    plt.title("Local SHAP Explanation — One Driver/Race")
    plt.tight_layout()
    plt.show()


## 15. Inference function

In [ ]:
def predict_podium(record, model, preprocessor, raw_feature_cols, threshold):
    if isinstance(record, dict):
        row = pd.DataFrame([record])
    elif isinstance(record, pd.Series):
        row = record.to_frame().T
    elif isinstance(record, pd.DataFrame):
        row = record.copy()
    else:
        raise TypeError("record must be dict, Series, or DataFrame")

    for c in raw_feature_cols:
        if c not in row.columns:
            row[c] = np.nan

    row = row[raw_feature_cols]
    X_new = np.asarray(
        preprocessor.transform(row),
        dtype=np.float32
    )

    prob = float(
        model.predict(
            X_new,
            verbose=0
        ).ravel()[0]
    )

    return {
        "podium_probability": prob,
        "podium_prediction": int(prob >= threshold),
        "threshold": float(threshold),
    }

if DATA_READY:
    complete_record = X_test_raw.iloc[0].copy()

    print("Complete record prediction:")
    print(
        predict_podium(
            complete_record,
            best_model,
            preprocessor,
            feature_cols,
            best_threshold
        )
    )

    partially_missing_record = complete_record.copy()

    num_to_blank = max(1, min(3, len(feature_cols)))
    for c in feature_cols[:num_to_blank]:
        partially_missing_record[c] = np.nan

    print("\nPartially-missing record prediction:")
    print("Missing fields:", feature_cols[:num_to_blank])

    print(
        predict_podium(
            partially_missing_record,
            best_model,
            preprocessor,
            feature_cols,
            best_threshold
        )
    )


## 16. Final interpretation checklist

Before submission, record the actual executed results:

- Which NN trial had the best **validation PR-AUC**?
- What validation-selected classification threshold was used?
- What were train / validation / unseen-test ROC-AUC, PR-AUC, and F1?
- Which feature-group removal caused the largest drop in PR-AUC?
- Which transformed features ranked highest by permutation importance?
- What did the global SHAP summary show?
- For one local prediction, which features pushed the prediction up/down?
- Explicitly state that SHAP/permutation importance reflects **model reliance, not causality**.

## Final team consistency check

Before merging/submitting:
1. Person 2's `modeling_table` must be the data source.
2. This notebook must use Person 2's exact `ALLOWED_FEATURES`.
3. Person 3's statistical models and this NN must use the same rows and temporal split.
4. Hyperparameters and threshold choices must use train/validation only.
5. The unseen test set must not be used for tuning.
6. The final Kaggle notebook must execute top-to-bottom with Run All.
